# Chapter 32 — Headless Pi

**Companion to *Pi Agents*** · [`Pi Agents` chapter 32](https://programmer.ie/books/pi/32-chapter/)

| | |
|---|---|
| Chapter | `32-chapter.md` · weight 32 · `/books/pi/32-chapter/` |
| Notebook | `notebooks/pi/32-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

With a failed response, is the failure an exit status or only in the stream — and
does `--tools` stop `bash`?

The chapter's point: print mode is a Unix program (exit status, stderr), JSON mode
is a stream (exit 0, failure recorded in the stream), and `--tools` is a real
allowlist — a `bash` call the model makes when `bash` is not in the list gets a
failed result and nothing runs.


## What this notebook establishes

- Print mode: the whole interface is the final text on stdout; a **failed** final response exits non-zero with the message on **stderr** and nothing on stdout.
- JSON mode: the same failure is in the event stream (`stopReason: error`) and the **exit status is 0**.
- With stdin and stdout redirected and no mode given, Pi **chooses print mode**.
- **Piped stdin is prepended** to the first prompt.
- `@path` includes a text file in the first prompt; `--` lets a prompt begin with a dash.
- `--tools` is a **real allowlist**: a `bash` call gets a failed result and **nothing runs**.
- `--no-session` writes no session file; without it a run against a real directory does.

## What this notebook does **not** establish

- **The four-tool claim is about built-ins.** As of 1.0.4 `--tools` no longer removes MCP tools, so `--no-mcp` is part of the command, not a footnote.
- **The model is scripted** — this measures the CLI's plumbing, not model behaviour.
- **No real credential or network** is used.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the shipped binary as a child process with a scripted model, offline.
* **Evidence scope:** `shipped_binary`. The CLI's contract is the real one.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(32))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/32-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: print vs JSON on the same failure

In [3]:
HEADLESS = pinb.driver_source("ch32-headless.ts")
headless = pinb.driver(HEADLESS, label="ch32-headless", timeout=420)
p, j = headless["print"], headless["json"]
print(pinb.md_table(
    ["run", "exit", "stdout", "stderr"],
    [["print success", p["okExit"], repr(p["okStdout"]), "-"],
     ["print failure", p["badExit"], "(empty)" if p["badStdoutEmpty"] else "(non-empty)", "has message" if p["badStderrHasMessage"] else "-"],
     ["JSON failure", j["badExit"], f"stopReason={j['stopReason']}", "-"]],
))

| run | exit | stdout | stderr |
|---|---|---|---|
| print success | 0 | 'the final answer\n' | - |
| print failure | 1 | (empty) | has message |
| JSON failure | 0 | stopReason=error | - |


In [4]:
pinb.show_checks([
    pinb.check("print success exits 0 with the answer on stdout",
               p["okExit"] == 0 and p["okStdout"] == "the final answer\n", "exit 0, answer"),
    pinb.check("print failure exits non-zero and writes nothing to stdout",
               p["badExit"] != 0 and p["badStdoutEmpty"], "non-zero, stdout empty"),
    pinb.check("the failure message is on stderr",
               p["badStderrHasMessage"], "message on stderr"),
    pinb.check("JSON failure exits 0 with the failure in the stream",
               j["badExit"] == 0 and j["stopReason"] == "error", "exit 0, stopReason error"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------  ------------------------
PASS  print success exits 0 with the answer on stdout  exit 0, answer
PASS  print failure exits non-zero and writes nothing to stdout  non-zero, stdout empty
PASS  the failure message is on stderr  message on stderr
PASS  JSON failure exits 0 with the failure in the stream  exit 0, stopReason error

4/4 assertions held.


## Prompt inputs: implicit print, piped stdin, `@path`, `--`

In [5]:
i, piped, at, dash = headless["implicit"], headless["piped"], headless["atPath"], headless["dash"]
print(pinb.md_table(
    ["input", "observed"],
    [["implicit print mode", repr(i["stdout"])],
     ["piped stdin first", "yes" if piped["addedBeforePrompt"] else "no"],
     ["@path included", "yes" if at["includesMarker"] else "no"],
     ["-- prompt kept", "yes" if dash["includesDashPrompt"] else "no"]],
))

| input | observed |
|---|---|
| implicit print mode | 'implicit\n' |
| piped stdin first | yes |
| @path included | yes |
| -- prompt kept | yes |


In [6]:
pinb.show_checks([
    pinb.check("redirected, no mode given: Pi chooses print mode",
               i["exit"] == 0 and i["stdout"] == "implicit\n", "implicit print"),
    pinb.check("piped stdin comes before the prompt", piped["addedBeforePrompt"], "stdin first"),
    pinb.check("@path includes the file", at["includesMarker"], "marker present"),
    pinb.check("-- lets a prompt begin with a dash", dash["includesDashPrompt"], "dash prompt kept"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  ----------------
PASS  redirected, no mode given: Pi chooses print mode  implicit print
PASS  piped stdin comes before the prompt  stdin first
PASS  @path includes the file  marker present
PASS  -- lets a prompt begin with a dash  dash prompt kept

4/4 assertions held.


## `--tools` is a real allowlist

In [7]:
n, d = headless["narrowed"], headless["defaultTools"]
print(pinb.md_table(
    ["run", "bash result isError?", "ran.flag created?"],
    [["--tools read,grep,find,ls", n["isError"], "yes" if n["ranFlagExists"] else "no"],
     ["default tools", "n/a", "yes" if d["ranFlagExists"] else "no"]],
))

| run | bash result isError? | ran.flag created? |
|---|---|---|
| --tools read,grep,find,ls | True | no |
| default tools | n/a | yes |


In [8]:
pinb.show_checks([
    pinb.check("--tools makes a bash request a failed result",
               n["isError"] is True, "isError true"),
    pinb.check("and nothing runs (no ran.flag)",
               not n["ranFlagExists"], "no side effect"),
    pinb.check("the default tool set can run bash",
               d["ranFlagExists"], "ran.flag created"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------  ----------------
PASS  --tools makes a bash request a failed result  isError true
PASS  and nothing runs (no ran.flag)  no side effect
PASS  the default tool set can run bash  ran.flag created

3/3 assertions held.


## Sessions

In [9]:
s = headless["session"]
print(pinb.md_table(
    ["run", "session file written?"],
    [["--no-session (default in this harness)", "no" if s["noSessionDir"] else "yes"],
     ["session: true", f"{s['keptSessionFiles']} .jsonl file(s)"]],
))

| run | session file written? |
|---|---|
| --no-session (default in this harness) | no |
| session: true | 1 .jsonl file(s) |


In [10]:
pinb.show_checks([
    pinb.check("a run with no session writes no session directory",
               s["noSessionDir"], "no sessions dir"),
    pinb.check("a session run writes exactly one .jsonl",
               s["keptSessionFiles"] == 1, "one file"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------  ---------------
PASS  a run with no session writes no session directory  no sessions dir
PASS  a session run writes exactly one .jsonl  one file

2/2 assertions held.


## The chapter's own tests

All ten shipped-binary tests.

## The chapter's canonical evidence (shipped binary)

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = 'ch32-headless/headless.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=420)
pinb.show_tap(tap, files='ch32-headless/headless.test.ts', show="mode")

$ node --test ch32-headless/headless.test.ts
  PASS  10 passed, 0 failed, 0 skipped  in 3501 ms

  tests matching 'mode':
    ok   print mode: the whole interface is the final text on stdout, and a successful run exits 0
    ok   print mode: a failed final response exits non-zero, with the message on stderr and nothing on stdout
    ok   JSON mode: the same failure is in the event stream and the exit status is 0
    ok   with stdin and stdout redirected and no mode given, Pi chooses print mode
    ok   --tools narrows the agent: a model that asks for bash gets a failed result, and nothing runs


## Your turn: predict, then run

**Predict first.** `--tools read,grep,find,ls` blocks `bash`. Does it also block
`edit` and `write`?

**Then change one input.** Add `--no-mcp` to the `--tools` run. What changes at
1.0.4, where `--tools` no longer removes MCP tools?

**Predict the boundary.** Print mode exits non-zero on a failure. What exit status
does a *successful* run with a non-empty `errorMessage` on the message return?

In [12]:
print("Predictions:")
print("  1. --tools read,grep,find,ls also blocks edit and write - they are not in the list.")
print("  2. --no-mcp removes MCP tools; --tools alone keeps them as of 1.0.4.")
print("  3. A successful run exits 0 regardless of a message errorMessage.")
print()
print("Observed above:")
print(f"  print: ok exit={p['okExit']}, bad exit={p['badExit']}, bad stdout empty={p['badStdoutEmpty']}")
print(f"  JSON failure: exit={j['badExit']}, stopReason={j['stopReason']}")
print(f"  --tools: isError={n['isError']}, ran.flag={n['ranFlagExists']}")
print()
assert p["okExit"] == 0 and p["badExit"] != 0 and p["badStdoutEmpty"]
assert j["badExit"] == 0 and j["stopReason"] == "error"
assert n["isError"] is True and not n["ranFlagExists"] and d["ranFlagExists"]
print("held: print is a Unix program, JSON is a stream, --tools is a real allowlist")

Predictions:
  1. --tools read,grep,find,ls also blocks edit and write - they are not in the list.
  2. --no-mcp removes MCP tools; --tools alone keeps them as of 1.0.4.
  3. A successful run exits 0 regardless of a message errorMessage.

Observed above:
  print: ok exit=0, bad exit=1, bad stdout empty=True
  JSON failure: exit=0, stopReason=error
  --tools: isError=True, ran.flag=False

held: print is a Unix program, JSON is a stream, --tools is a real allowlist


## Interpretation

Headless Pi has three contracts, and the chapter shows each:

| Mode | Answer | Failure | Sessions |
|---|---|---|---|
| **print** | Final text on stdout | Non-zero exit, message on stderr | `--no-session` or default |
| **JSON** | Event stream | Exit 0, `stopReason: error` in the stream | as configured |
| **implicit** | print (when redirected, no flag) | as print | as configured |

Inputs:
* **Piped stdin** is prepended to the first prompt.
* **`@path`** includes a file's contents.
* **`--`** ends option parsing so a prompt can start with a dash.

The allowlist:
* `--tools read,grep,find,ls` — a `bash` call gets a failed result and nothing runs.
* As of **1.0.4**, `--tools` no longer removes MCP tools; add `--no-mcp`.

Practical rules:
1. **Print for scripts** — the exit status is the contract.
2. **JSON for observers** — the failure is a record, not a status.
3. **`--tools` for capability control** — it is an allowlist, not a hint.
4. **`--no-session` for one-shot work** — no file is written.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the shipped binary as a child process with a scripted model, offline.

**Evidence scope:** `shipped_binary`. The CLI's contract is the real one.

### What was read or run

- **Ran** `drivers/ch32-headless.ts`: nine shipped-binary runs (print success/failure, JSON failure, implicit mode, piped stdin, `@path`, `--`, `--tools`, default tools, sessions).
- **Ran** `ch32-headless/headless.test.ts` (10 tests).
- **Read** `examples/evidence.json`, chapter 32 rows (10 claims).

### Limitations

- **As of 1.0.4 `--tools` no longer removes MCP tools** — `--no-mcp` is part of the command.
- **The model is scripted** — the CLI's plumbing is exercised, not model behaviour.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
